# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIRˆ2 dataset using the `mlcroissant` library, following the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data via `RecordSet` entities. Each `RecordSet` is uniquely referenced by its `@id`.

Let's enumerate all record sets available in the dataset and discover their fields.

In [ ]:
# Inspect available record sets and fields by their @id
record_sets = []
for record_set in dataset.record_sets:
    print(f"RecordSet: {record_set['@id']} - {record_set.get('name','Unnamed RecordSet')}")
    fields = record_set.get('field', [])
    print("  Fields:")
    for field in fields:
        print(f"    - {field['@id']} (name: {field.get('name','')}, type: {field.get('dataType','')})")
    record_sets.append(record_set['@id'])
if not record_sets:
    print("No record sets found in the schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We use the `recordSet` `@id` to reference the record set and load it into a DataFrame. Replace the placeholders appropriately if you want to analyze another record set.

In [ ]:
# Extract data from all available record sets
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found in record set {record_set_id}.")

# For demonstration, select the first available record set for further analysis
record_set_id = record_sets[0] if record_sets else None
if record_set_id and record_set_id in dataframes:
    print(f"Using RecordSet {record_set_id} for further analysis.")
else:
    print("No valid record set for further analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We demonstrate EDA using a numeric field and a categorical field, referencing all fields explicitly by their `@id`.

In [ ]:
# Example EDA on first available record set
if record_set_id and record_set_id in dataframes:
    df = dataframes[record_set_id]

    # Identify numeric fields by inspecting their @id and dataType
    # For demonstration, assume 'cr:field_Age' is a numeric field and 'cr:field_Sex' is categorical.
    numeric_field_id = None
    group_field_id = None
    record_set_obj = None
    for rs in dataset.record_sets:
        if rs['@id'] == record_set_id:
            record_set_obj = rs
            break
    if record_set_obj:
        for field in record_set_obj.get('field', []):
            if 'dataType' in field and field['dataType'] in ['schema:Integer', 'schema:Float', 'schema:Number']:
                numeric_field_id = field['@id']
                print(f"Selected numeric field: {numeric_field_id}")
                break
        for field in record_set_obj.get('field', []):
            if 'dataType' in field and field['dataType'] == 'schema:Text':
                group_field_id = field['@id']
                print(f"Selected categorical (group) field: {group_field_id}")
                break
    # Proceed if a numeric field is found
    if numeric_field_id and numeric_field_id in df.columns:
        threshold = 50
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Group by the group_field
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No record set DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

This section plots the distribution of the filtered numeric field and displays group-wise averages.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and record_set_id in dataframes and numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold].copy()

    # Plot histogram of numeric_field
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Plot group-wise averages if group_field_id is available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_means)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("Visualization skipped: Required fields not found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrates step-by-step exploration of the FAIRˆ2 clinical colorectal cancer dataset using the Croissant schema and `mlcroissant` tools. 

**Key takeaways:**
- Explicit use of `@id` ensures clarity and reproducibility across all entity references.
- `mlcroissant` efficiently loads metadata and tabular data conforming to FAIR principles.
- Data processing and visualization steps enable clinical and molecular insight into the record sets provided.

You may extend this notebook further by exploring additional record sets, fields, and applying more advanced analyses as needed.